In [2]:
using Lux, DiffEqFlux, OrdinaryDiffEq, Plots, Printf, Statistics
using ComponentArrays
using Optimization, OptimizationOptimisers
using Enzyme
using Random
using StaticArrays
using SciMLSensitivity
using SciMLStructures

In [3]:
Enzyme.API.looseTypeAnalysis!(true)

In [4]:

struct PhaseEnergies
    G::AbstractVector
    Ea::AbstractMatrix
    barriers::AbstractMatrix
    function PhaseEnergies(G::AbstractVector, forward_Ea::AbstractMatrix)
        n = length(G)
        @assert size(forward_Ea) == (n, n)
        deltaG = ΔG(G)
        barriers = [i == j ? 0 : (deltaG[i, j] > 0 ? deltaG[i, j] + forward_Ea[i, j] : forward_Ea[i, j])
                for j in 1:n, i in 1:n]
        new(G, forward_Ea, Matrix{eltype(G)}(barriers))
    end
end

n_phases(pe::PhaseEnergies) = length(pe.G)
ΔG(gmat::AbstractVector) = [gmat[j] - gmat[i] for j in eachindex(gmat), i in eachindex(gmat)]
ΔG(pe::PhaseEnergies) = ΔG(pe.G)

ΔG (generic function with 2 methods)

In [5]:
kb = 8.617e-5 #eV/K

function flow_coefficient(type, effecting_nums, decay_coefficient)
"""
Inputs:
    type: type of decay (exponential or linear)
    effecting_nums: number of effecting layers on top of the current layer
Output:
    flow_coefficients: array of flow coefficients for each layers
Checkstat: Checked
"""
    effecting_nums = round(Int, effecting_nums)
    fcoeff = ones(effecting_nums)
    for j in 1:effecting_nums
        if type == "exponential"
            fcoeff[j] = exp(-decay_coefficient * j)
        elseif type == "linear"
            fcoeff[j] = (1 - decay_coefficient * j)
        end
    end
    return vcat(zeros(effecting_nums-1), fcoeff) #Add 0s for reverse
end

arrhenius_rate(pe::PhaseEnergies, T::Real=300) = arrhenius_rate(Array(pe.barriers), T)

# move calculation to helper fcn to make AD easier

function arrhenius_rate(barriers, T=300)
"""
Inputs:
    barriers: array of barriers
    T: temperature
Output:
    K: array of rate constants
Checkstat: Checked
"""
    kb = 8.617e-5 #eV/K
    A = 1.0 # Arrhenius prefactor
    K = A * exp.(-barriers ./ (kb * T))
    # Adjust the diagonal elements
    for i in axes(K,1)
        K[i, i] =  -1 * sum(K[i, [1:i-1; i+1:end]])
    end
    return K
end

arrhenius_rate (generic function with 4 methods)

In [9]:
function meshgrid(x, y)
    """
    To build the meshgrids for the x and y values (Temperature and Flow Rate)
    """
        x_grid = repeat(reshape(x, 1, :), length(y), 1)
        y_grid = repeat(y, 1, length(x))
        return x_grid, y_grid
end

function deposition_rates!(dc, c, p, t)
"""
Checkstat: Checked
"""
    # Unpack parameters
    fcoeff = p.fcoeff
    K = p.K
    j0 = p.j0
    j = p.j
    dt = p.dt
    num_steps = Int(p.num_steps)
    num_layers = Int(p.num_layers)
    # Calculate deposition rates
    j = floor(Int, t / 0.5) + 1
    f = reverse(fcoeff[j: num_layers+j-1])
    dc .= c .* f * K
    if j != j0
        c[j+1, 1] = 1.0
        j = j0
    end
end

function simulate_deposition(flow_rate, T, barriers::Matrix, para_sim, decay_constant = 0.00001, final_step=true)
    num_steps, num_layers, dt = para_sim
    # Initialize existing_layers as a 2D array
    decay_coefficients = decay_constant * flow_rate
    fcoeff = flow_coefficient("exponential", num_layers, decay_coefficients)
    n = size(barriers, 1)
    c0 = zeros(num_layers, n)
    c0[1, 1] = 1.0
    K = arrhenius_rate(barriers, T)
    j = 0
    j0 = 0
    p = (fcoeff, K, j0, j, dt, num_steps, num_layers)
    p = NamedTuple{(:fcoeff, :K, :j0, :j, :dt, :num_steps, :num_layers)}(p)
    p = ComponentArray(p)
    tspan = (0.0, (num_steps-1) * dt)
    prob = ODEProblem(deposition_rates!, c0, tspan, p)
    if final_step
        sol = solve(prob, Euler(), save_everystep = false, dt=dt)
        #sol = solve(prob, Euler(), dt = 0.5)
        return Array(sol.u[end])
    else
        sol = solve(prob, Euler(), saveat = 0.5, dt = dt)
        return sol.u
    end
end



simulate_deposition (generic function with 3 methods)

In [10]:
G_values = [-5.10, -5.97, -5.85]
Ea_constants = [0.00 1.0 0.36; 1.0 0.00 0.38; 0.36 0.38 0.00]
gr()
rng = Xoshiro(0)
pe = PhaseEnergies(G_values, Ea_constants)
n = n_phases(pe)
display(pe.barriers)
T = 300.0
flow_rate = 1.5
threshold = 0.3 # Threshold for most preferable state
t = 600 # seconds
dt = 0.5 # seconds
num_steps = round(Int, t/dt)
num_layers = floor(Int, t/0.5)+1
para_sim = num_steps, num_layers, dt
phase_names = ["x", "β", "κ"]
compositions_all = simulate_deposition(flow_rate, T, pe.barriers, para_sim, 0.0022)
compositions_all = Array(compositions_all)

3×3 Matrix{Float64}:
 0.0   1.0   0.36
 1.87  0.0   0.5
 1.11  0.38  0.0

1201×3 Matrix{Float64}:
 0.999867  4.06414e-9   0.000132844
 0.999867  4.06361e-9   0.000132836
 0.999867  4.06308e-9   0.000132827
 0.999867  4.06256e-9   0.000132818
 0.999867  4.06203e-9   0.00013281
 0.999867  4.0615e-9    0.000132801
 0.999867  4.06096e-9   0.000132792
 0.999867  4.06043e-9   0.000132784
 0.999867  4.05989e-9   0.000132775
 0.999867  4.05935e-9   0.000132766
 ⋮                      
 0.999997  1.89142e-12  3.09297e-6
 0.999997  1.35547e-12  2.65548e-6
 0.999998  9.06636e-13  2.21655e-6
 0.999998  5.45784e-13  1.77616e-6
 0.999999  2.738e-13    1.33432e-6
 0.999999  9.1576e-14   8.91011e-7
 1.0       7.89999e-18  4.46241e-7
 1.0       0.0          0.0
 1.0       0.0          0.0

In [11]:
inputs = [T, flow_rate]
input_size = length(inputs)  # Replace with the actual size of `inputs` if it's not a 1D vector
barrier_size = (n ^ 2)
fcoeff_size = 1 #sigmoid 0~1 #
precoeff_size = 0
output_size = barrier_size + fcoeff_size + precoeff_size
nn = Chain(
    Dense(input_size, input_size*3*n, tanh),
    Dense(input_size*3*n, output_size*2, tanh),
    Dense(output_size*2, output_size, sigmoid)
)

Chain(
    layer_1 = Dense(2 => 18, tanh),     # 54 parameters
    layer_2 = Dense(18 => 20, tanh),    # 380 parameters
    layer_3 = Dense(20 => 10, σ),       # 210 parameters
)         # Total: 644 parameters,
          #        plus 0 states.

In [12]:
u, st = Lux.setup(rng, nn)

((layer_1 = (weight = Float32[-1.8019577 0.46916437; -0.18273845 0.7021897; … ; 1.5201496 -0.27108315; -0.69673556 -0.6062841], bias = Float32[-0.522484, -0.6805993, -0.21060704, 0.50937545, 0.33639288, 0.22010256, -0.12450862, 0.3884359, 0.5799375, 0.39842856, -0.6958851, 0.07831879, -0.30917966, -0.5286344, 0.032922927, -0.07239345, 0.30830613, 0.17590544]), layer_2 = (weight = Float32[0.6362607 0.513106 … -0.5238187 0.34960684; 0.62408286 0.5794506 … 0.31360275 0.21403408; … ; 0.5591235 -0.629305 … 0.326576 -0.40669045; -0.4567112 -0.15265568 … 0.045966778 -0.47867388], bias = Float32[0.08092231, -0.13791399, -0.19817856, -0.009384923, -0.098436914, -0.108139515, 0.12382061, -0.014105763, 0.16400078, -0.20324503, 0.06229071, -0.018715758, -0.15202452, -0.13853726, -0.10645687, 0.12905023, 0.03892694, -0.1408768, 0.16721667, 0.129089]), layer_3 = (weight = Float32[-0.16629656 0.22899275 … -0.35004017 -0.32747597; -0.37176996 -0.0035260152 … 0.13210998 -0.304247; … ; 0.22372834 -0.080

In [13]:
function predict_neuralode(u)
    # Get parameters from the neural network
    inputs = [T, flow_rate]
    output, outst = nn(inputs, u, st)

    # Segregate the output
    pp_barrier = output[1:barrier_size]
    p_barrier = reshape(pp_barrier, (n, n))
    p_fcoeff = output[barrier_size+1:barrier_size+fcoeff_size]
    # Amorphous phase goes to zero
    nn_output = (p_barrier, p_fcoeff)
    predicted_composition = simulate_deposition(flow_rate, T, p_barrier, para_sim, p_fcoeff[1])
    return Array(predicted_composition)
end

function loss_neuralode(p)
    pred = predict_neuralode(p)
    loss = sum(abs2, compositions_all .- pred)
    return loss, pred
end

loss_neuralode (generic function with 1 method)

In [14]:
callback = function (state::Optimization.OptimizationState, loss_value::Float64; doplot = false)
    p = state.u
    l, pred = loss_neuralode(p)
    println(l)
    if doplot
        pred_avg = mean(pred, dims=1)
        #pred_avg = reshape(pred_avg, (3, 21))
        #plot the three phases from ode_data_avg and pred_avg
        plt = scatter(tsteps, ode_data_avg[1], label = "Phase 1 Data", color = :blue)
        scatter!(plt, tsteps, ode_data_avg[2], label = "Phase 2 Data", color = :red)
        scatter!(plt, tsteps, ode_data_avg[3], label = "Phase 3 Data", color = :green)
        scatter!(plt, tsteps, pred_avg[1], label = "Phase 1 Prediction", color = :blue, shape = :cross)
        scatter!(plt, tsteps, pred_avg[2], label = "Phase 2 Prediction", color = :red, shape = :cross)
        scatter!(plt, tsteps, pred_avg[3], label = "Phase 3 Prediction", color = :green, shape = :cross)
        display(plot(plt))
        savefig(plt, "training_$timestamp.svg")
    end
    return false
end

#15 (generic function with 1 method)

In [15]:
pinit = ComponentArray(u)
#callback(pinit, loss_neuralode(compositions_all, pinit)...)

adtype = Optimization.AutoEnzyme(; mode=set_runtime_activity(Reverse))

optf = Optimization.OptimizationFunction((x,_) -> loss_neuralode(x), adtype)
optprob = Optimization.OptimizationProblem(optf, pinit)

result_neuralode = Optimization.solve(
    optprob, OptimizationOptimisers.Adam(0.02); callback = callback, maxiters = 5)

┌ Warning: Mixed-Precision `matmul_cpu_fallback!` detected and Octavian.jl cannot be used for this set of inputs (C [Matrix{Float64}]: A [Base.ReshapedArray{Float32, 2, SubArray{Float32, 1, Vector{Float32}, Tuple{UnitRange{Int64}}, true}, Tuple{}}] x B [Matrix{Float64}]). Converting to common type to to attempt to use BLAS. This may be slow.
└ @ LuxLib.Impl C:\Users\heyye\.julia\packages\LuxLib\kH9PB\src\impl\matmul.jl:148


AssertionError: AssertionError: Enzyme Internal Error: did not have sret when expected
f=; Function Attrs: alwaysinline mustprogress
define internal "enzymejl_parmtype"="1722898766864" "enzymejl_parmtype_ref"="1" { { {} addrspace(10)*, {} addrspace(10)*, {} addrspace(10)*, {} addrspace(10)*, {} addrspace(10)*, {} addrspace(10)*, {} addrspace(10)* }, { {} addrspace(10)*, {} addrspace(10)*, {} addrspace(10)*, {} addrspace(10)*, { { [1 x [4 x {} addrspace(10)*]], [1 x i8] }, {} addrspace(10)*, {} addrspace(10)*, {} addrspace(10)*, i8, [5 x {} addrspace(10)*], i8 }, i8, i64, {} addrspace(10)*, i32 }, { {} addrspace(10)*, {} addrspace(10)*, {} addrspace(10)*, {} addrspace(10)*, { { [1 x [4 x {} addrspace(10)*]], [1 x i8] }, {} addrspace(10)*, {} addrspace(10)*, {} addrspace(10)*, i8, [5 x {} addrspace(10)*], i8 }, i8, i64, {} addrspace(10)*, i32 } } @augmented_julia_solve_18742_inner.1({ i8, double } "enzyme_type"="{[0]:Integer, [8]:Float@double}" "enzymejl_parmtype"="1724293574288" "enzyme